# Task 4.2. Tinh chỉnh GGNN trên Graph B

| | |
|---|---|
| **Mục đích** | So sánh từng yếu tố với GGNN-B mặc định, chọn bằng validation, rồi báo test cho cấu hình đã chọn |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` (version 1) |
| **Đầu ra** | `results_4_2_ggnn_tuning/<ID>/` (`results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md`, metadata, trọng số), `frozen_selection.json`, `results_4_2_ggnn_final/` (kết quả test), `results_4_2_ggnn_sensitivity/` (test của các cấu hình không được chọn, chỉ để phân tích độ nhạy), thư mục `submission_4_2` cùng tệp zip để nộp |
| **Máy chạy** | Kaggle GPU (T4 hoặc P100), bật Internet |

Hai protocol `project_wise`, `seen_project`, seed 42, 43, 44, cùng 7 cấu hình như 4.1b để so sánh công bằng (B0, E1, D1, W1, H1, L1, N1). Dùng chung model, trainer, metric với 4.1a. Không sửa dữ liệu, vocabulary, nhãn hoặc split. Test chỉ dùng sau khi cố định cấu hình. Chạy "Run All" một lần là đủ: tuning, chọn cấu hình, test, biểu đồ, phân tích độ nhạy, đóng gói.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
APPROVED_BRANCH = "feature/4_2_ggnn_tuning"
APPROVED_COMMIT = "ba5656eb2ea93b8ea31a066d16128b5c3bd63cc7"  # SHA đầy đủ của commit chứa mã nguồn
DATASET_ID = "vtnmai/diversevul-9077-processed-graph-data/versions/1"
APPROVED_CONFIG_IDS = ["B0", "E1", "D1", "W1", "H1", "L1", "N1"]  # giống 4.1b
RUN_TUNING = True
RUN_FINAL_TEST = True
RUN_SENSITIVITY = True  # chấm test cho 6 cấu hình còn lại, chỉ để phân tích độ nhạy
# Phục hồi sau khi phiên chết: gắn output cũ làm dataset rồi trỏ tới thư mục tuning đã lưu.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_2_ggnn_tuning
OUT_DIR = Path("/kaggle/working/results_4_2_ggnn_tuning")
FINAL_OUT_DIR = Path("/kaggle/working/results_4_2_ggnn_final")
SENS_DIR = Path("/kaggle/working/results_4_2_ggnn_sensitivity")
FIG_DIR = Path("/kaggle/working/figures_4_2")
SUBMISSION_DIR = Path("/kaggle/working/submission_4_2")
ACTIVE = RUN_TUNING or RUN_FINAL_TEST
if ACTIVE:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
    if not re.fullmatch(r"vtnmai/diversevul-9077-processed-graph-data/versions/[1-9][0-9]*", DATASET_ID):
        raise ValueError("Ghi phiên bản Kaggle dataset đã gắn vào notebook.")
REPO_DIR = Path("/tmp/g42") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR

## 1. Tải mã nguồn và ghi phiên bản

Clone nhánh đã duyệt, checkout đúng SHA. Không `git pull`, không cài thư viện trong notebook. Giữ output notebook để lưu nhật ký chạy.

In [ ]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training.gcn_tuning import (
        BASELINE, CANDIDATES, GGNN_HYPOTHESES, GGNN_B, PROTOCOLS, SEEDS,
        configuration, code_identity, environment, freeze_winner, write_json,
        reuse_candidate, read_frozen, validate_context, validate_final,
    )
    from src.graph.processed import ids_hash, load_schema
    SCOPE = GGNN_B  # (model, graph) = ("ggnn", "graph_b")
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings (T4 hoặc P100)."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    print("Revision:", actual, "Environment:", environment(), flush=True)


def run_script(arguments, log_path):
    cmd = [sys.executable, "scripts/train_gnn.py", *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")

## 2. Dữ liệu và cấu hình

Mô hình là GGNN (3 loại cạnh × 2 chiều, GRUCell) trên Graph B (AST, CFG, DATA_DEP). Mỗi biến thể chỉ đổi một yếu tố so với B0: E1 dừng theo validation loss, D1 dropout 0,4, W1 weight decay 1e-3, H1 dim 64, L1 hai bước lan truyền, N1 tối đa 1.000 node. Các cấu hình được so bằng trung bình Macro-F1 validation `project_wise`, `seen_project` là đối chứng.

B0 của 4.1a là tham chiếu lịch sử. Notebook này chạy lại B0 trong cùng môi trường để so trực tiếp.

In [ ]:
if ACTIVE:
    found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
             if all((p.parent / protocol / "vocab.json").is_file() for protocol in PROTOCOLS)]
    assert len(found) == 1, "Gắn đúng một phiên bản dataset processed chính thức."
    PROCESSED = found[0]
    metadata = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    assert metadata.get("spec_version") in {"1.1", "1.2"}
    assert metadata.get("n_valid_samples") == 7799
    assert sorted(load_schema(REPO_DIR / "configs/graph/schema_spec.yaml")["cwe_mapping"].values()) == list(range(23))
    from collections import Counter
    expected = {"project_wise": {"train": 6101, "validation": 762, "test": 936},
                "seen_project": {"train": 6272, "validation": 764, "test": 763}}
    for protocol in PROTOCOLS:
        with (PROCESSED / protocol / "manifest.csv").open(encoding="utf-8", newline="") as handle:
            manifest = list(csv.DictReader(handle))
        counts = Counter(row["split"] for row in manifest)
        assert len({row["sample_id"] for row in manifest}) == len(manifest)
        assert all(0 <= int(row["y"]) < 23 for row in manifest)
        vocab = json.loads((PROCESSED / protocol / "vocab.json").read_text(encoding="utf-8"))
        assert isinstance(vocab["k"], int) and vocab["k"] > 0
        for name in ("node_type", "token"):
            assert sorted(vocab[name].values()) == list(range(len(vocab[name])))
            assert vocab[name].get("<PAD>") == 0 and vocab[name].get("<UNK>") == 1
        assert vocab["token"].get("<MASK>") == 2
        assert vocab.get("train_ids_sha256") == ids_hash(row["sample_id"] for row in manifest if row["split"] == "train")
        assert dict(counts) == expected[protocol]
        for split in expected[protocol]:
            assert (PROCESSED / protocol / f"graph_b_{split}.pt").is_file()
    print("Dataset:", DATASET_ID, PROCESSED)
if RUN_TUNING:
    assert APPROVED_CONFIG_IDS and len(set(APPROVED_CONFIG_IDS)) == len(APPROVED_CONFIG_IDS)
    assert all(key in CANDIDATES for key in APPROVED_CONFIG_IDS)
    for key in APPROVED_CONFIG_IDS:
        cfg = configuration(key)
        changed = {k: v for k, v in cfg.items() if v != BASELINE[k]}
        assert len(changed) == (0 if key == "B0" else 1)
        print(key, changed, GGNN_HYPOTHESES[key], flush=True)
    print("Số cấu hình:", len(APPROVED_CONFIG_IDS), "Số lần train tối đa:", 6 * len(APPROVED_CONFIG_IDS))

## 3. Train chỉ validation

Chạy từng cấu hình chưa có output, không nạp test. Cấu hình đã hoàn tất chỉ được dùng lại sau khi kiểm tra đủ artifact, metadata, code, dataset và môi trường. Nếu phiên trước bị ngắt: gắn output cũ làm dataset, đặt `SAVED_RESULTS_ROOT`, chạy lại, các cấu hình hợp lệ được chép sang, không train lại. Không tự tiếp tục optimizer của lần train bị ngắt: thư mục dở dang gây lỗi rõ ràng, đổi `OUT_DIR` sang thư mục mới.

In [ ]:
if RUN_TUNING:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    plan = {
        "code": code_identity(REPO_DIR), "dataset_id": DATASET_ID, "model": SCOPE[0], "graph": SCOPE[1],
        "configurations": {key: configuration(key) for key in APPROVED_CONFIG_IDS},
        "hypotheses": {key: GGNN_HYPOTHESES[key] for key in APPROVED_CONFIG_IDS},
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS),
        "selection": "project_wise.validation.macro_f1.mean",
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
    }
    plan_path = OUT_DIR / "experiment_plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:
            raise ValueError("Kế hoạch đã lưu khác cấu hình hiện tại; không ghi đè.")
    else:
        with plan_path.open("x", encoding="utf-8") as handle:
            json.dump(plan, handle, ensure_ascii=False, indent=2)
    for key in APPROVED_CONFIG_IDS:
        target = OUT_DIR / key
        source = target if target.exists() else READ_ROOT / key
        if source.exists():
            reuse_candidate(source, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID, SCOPE)
            if source.resolve() != target.resolve():
                shutil.copytree(source, target)  # destination must be unused
        else:
            args = ["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                    "--models", SCOPE[0], "--graphs", SCOPE[1], "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--validation-only", "--save-checkpoints",
                    "--config-id", key, "--dataset-id", DATASET_ID]
            for name, value in configuration(key).items():
                if value is not None:
                    args += ["--" + name.replace("_", "-"), str(value)]
            run_script(args, OUT_DIR / f"{key}.log")
            reuse_candidate(target, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID, SCOPE)

## 4. Cố định lựa chọn bằng validation

Xếp hạng bằng trung bình Macro-F1 validation chưa làm tròn của `project_wise` trên ba seed. Khi bằng nhau ưu tiên B0, E1, D1, W1, H1, L1, N1. Dùng cùng cấu hình cho cả hai protocol. Nếu đã có lựa chọn thì chỉ nạp và xác minh, không thay bằng cấu hình khác. Không chọn theo test.

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    schema = REPO_DIR / "configs/graph/schema_spec.yaml"
    FROZEN_ROOT = OUT_DIR if RUN_TUNING else READ_ROOT
    FROZEN = FROZEN_ROOT / "frozen_selection.json"
    saved_frozen = READ_ROOT / "frozen_selection.json"
    if RUN_TUNING and not FROZEN.exists() and saved_frozen.exists():
        previous = read_frozen(saved_frozen, REPO_DIR, PROCESSED, schema, READ_ROOT, SCOPE)
        if {r["config_id"] for r in previous["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không tự chọn lại.")
        with FROZEN.open("xb") as handle:
            handle.write(saved_frozen.read_bytes())
    if FROZEN.exists():
        frozen = read_frozen(FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT, SCOPE)
        if RUN_TUNING and {r["config_id"] for r in frozen["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không ghi đè.")
    elif RUN_TUNING:
        frozen = freeze_winner([OUT_DIR / key for key in APPROVED_CONFIG_IDS], FROZEN, SCOPE)
    else:
        raise FileNotFoundError(f"Thiếu lựa chọn validation đã cố định: {FROZEN}")
    validate_context(frozen["winner"]["metadata"], REPO_DIR, PROCESSED, schema, DATASET_ID, SCOPE)
    print("Validation ranking:", frozen["ranking"], flush=True)
    print("Frozen configuration:", frozen["winner"]["config_id"], flush=True)

## 5. Kết quả validation: xếp hạng, đường học, chi phí

Các bảng và biểu đồ dưới đây chỉ dùng validation. `seen_project` chỉ để đối chứng, chọn cấu hình theo `project_wise`. Ở bước này chưa có số test nào.

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    WINNER = frozen["winner"]["config_id"]
    IDS = [r["config_id"] for r in frozen["ranking"]]  # đã xếp theo validation
    val = pd.concat([pd.read_csv(FROZEN_ROOT / k / "results.csv").assign(config=k) for k in IDS], ignore_index=True)
    hist = {}
    for k in IDS:
        for item in json.loads((FROZEN_ROOT / k / "training_history.json").read_text(encoding="utf-8")):
            protocol, _, _, seed = item["tag"].split("|")
            hist[(k, protocol, int(seed))] = pd.DataFrame(item["history"])

    def agg(frame, metric):
        g = frame.groupby(["config", "protocol"])[metric].agg(["mean", "std"]).unstack("protocol")
        return g.reindex(IDS)

    rank = pd.DataFrame(index=IDS)
    for protocol in PROTOCOLS:
        part = val[val.protocol == protocol].groupby("config")
        rank[f"{protocol} Macro-F1"] = part.macro_f1.apply(lambda s: f"{s.mean():.4f} ± {s.std(ddof=0):.4f}")
        rank[f"{protocol} Δ so với B0"] = part.macro_f1.mean() - part.macro_f1.mean()["B0"] if "B0" in IDS else np.nan
    rank["weighted-F1 (project_wise)"] = val[val.protocol == "project_wise"].groupby("config").weighted_f1.mean().round(4)
    rank["MCC (project_wise)"] = val[val.protocol == "project_wise"].groupby("config").mcc.mean().round(4)
    rank["epoch tốt nhất"] = val.groupby("config").best_epoch.mean().round(1)
    rank["giây/lần train"] = val.groupby("config").train_seconds.mean().round(0)
    rank["VRAM đỉnh (MB)"] = val.groupby("config").peak_gpu_mb.max().round(0)
    rank.index = [f"{k} *" if k == WINNER else k for k in rank.index]
    print("* = cấu hình được chọn theo validation project_wise")
    display(rank.round(4))
    rank.to_csv(FIG_DIR / "validation_ranking.csv")

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, protocol in zip(axes, PROTOCOLS):
        sub = val[val.protocol == protocol]
        means = sub.groupby("config").macro_f1.mean().reindex(IDS)
        stds = sub.groupby("config").macro_f1.std(ddof=0).reindex(IDS)
        colors = ["#d95f02" if k == WINNER else "#7570b3" for k in IDS]
        ax.bar(IDS, means, yerr=stds, color=colors, capsize=4)
        for i, k in enumerate(IDS):
            ax.scatter([i] * 3, sub[sub.config == k].macro_f1, color="black", s=10, zorder=3)
        if "B0" in means:
            ax.axhline(means["B0"], color="gray", ls="--", lw=1, label="B0")
        ax.set_title(f"Macro-F1 validation, {protocol}")
        ax.set_xlabel("Cấu hình (cam = được chọn)")
        ax.legend()
    axes[0].set_ylabel("Macro-F1 validation (trung bình ± độ lệch chuẩn, chấm = từng seed)")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "validation_macro_f1.png", bbox_inches="tight")
    plt.show()

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    compare = [k for k in ("B0", WINNER) if k in IDS]
    compare = list(dict.fromkeys(compare))
    colors = {"B0": "#1b9e77", WINNER: "#d95f02"}
    fig, axes = plt.subplots(2, len(PROTOCOLS), figsize=(12, 7), sharex="col")
    for j, protocol in enumerate(PROTOCOLS):
        for k in compare:
            for seed in SEEDS:
                h = hist[(k, protocol, seed)]
                label = k if seed == SEEDS[0] else None
                axes[0, j].plot(h.epoch, h.train_loss, color=colors[k], alpha=0.7, label=label)
                axes[1, j].plot(h.epoch, h.val_macro_f1, color=colors[k], alpha=0.7, label=label)
                best = val[(val.config == k) & (val.protocol == protocol) & (val.seed == seed)].best_epoch.iloc[0]
                axes[1, j].axvline(best, color=colors[k], ls=":", alpha=0.5)
        axes[0, j].set_title(f"Loss train, {protocol}")
        axes[1, j].set_title(f"Macro-F1 validation, {protocol} (đường chấm = epoch tốt nhất)")
        axes[1, j].set_xlabel("Epoch")
        axes[0, j].legend()
    axes[0, 0].set_ylabel("train loss")
    axes[1, 0].set_ylabel("Macro-F1 validation")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "learning_curves.png", bbox_inches="tight")
    plt.show()
    # Dấu hiệu học thuộc: loss train cuối so với Macro-F1 validation tốt nhất.
    rows = []
    for k in compare:
        for protocol in PROTOCOLS:
            last = [hist[(k, protocol, s)].train_loss.iloc[-1] for s in SEEDS]
            best = val[(val.config == k) & (val.protocol == protocol)].macro_f1
            rows.append({"config": k, "protocol": protocol, "train loss cuối": np.mean(last),
                         "Macro-F1 validation tốt nhất": best.mean()})
    display(pd.DataFrame(rows).round(4))

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    cost = val.groupby("config")[["train_seconds", "peak_gpu_mb", "n_params"]].mean().reindex(IDS)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].bar(cost.index, cost.train_seconds, color="#66a61e")
    axes[0].set_title("Giây huấn luyện mỗi lần (trung bình)")
    axes[1].bar(cost.index, cost.peak_gpu_mb, color="#e6ab02")
    axes[1].set_title("VRAM đỉnh (MB)")
    for ax in axes:
        ax.set_xlabel("Cấu hình")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "cost.png", bbox_inches="tight")
    plt.show()

## 6. Đánh giá test của cấu hình đã cố định

Chạy `--frozen-selection`: script xác minh lựa chọn rồi nạp trọng số đã lưu, không train lại, chỉ đọc `graph_b_test.pt`. `FINAL_OUT_DIR` phải là thư mục trống. Nếu đã có kết quả hợp lệ thì xác minh và dùng lại, không ghi đè.

In [ ]:
if RUN_FINAL_TEST:
    if FINAL_OUT_DIR.exists() and any(FINAL_OUT_DIR.iterdir()):
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT, SCOPE)
    else:
        run_script(["--processed-dir", str(PROCESSED), "--out-dir", str(FINAL_OUT_DIR),
                    "--frozen-selection", str(FROZEN), "--results-root", str(FROZEN_ROOT)],
                   FINAL_OUT_DIR.parent / (FINAL_OUT_DIR.name + ".log"))
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT, SCOPE)
    print((FINAL_OUT_DIR / "summary.md").read_text(encoding="utf-8"))

## 7. So sánh test với các mốc

Các mốc lấy từ kết quả đã commit trong repo: baseline token và GNN mặc định của 4.1a, GCN-A đã tinh chỉnh của 4.1b. GCN-B đã tinh chỉnh (4.1c) chưa có trong commit này nên chỉ có GCN-B mặc định. Nếu hiệu hai số nhỏ hơn độ lệch chuẩn lớn hơn giữa các seed thì không kết luận hơn hay kém.

In [ ]:
if RUN_FINAL_TEST:
    A = REPO_DIR / "reports/graph/4_1a_results"
    base = pd.read_csv(A / "baselines_results.csv")
    gnn = pd.read_csv(A / "gnn_results.csv")
    tuned = pd.read_csv(REPO_DIR / "reports/graph/4_1b_results/final/results.csv")
    mine = pd.read_csv(FINAL_OUT_DIR / "results.csv")
    parts = []
    for name, model in (("token_nodes", "token_nodes"), ("token_method", "token_method"),
                        ("random", "random"), ("majority", "majority")):
        parts.append(base[(base.split == "test") & (base.model == model)].assign(method=f"{name} (4.1a)"))
    for m in ("gcn", "ggnn"):
        for g in ("graph_a", "graph_b"):
            parts.append(gnn[(gnn.split == "test") & (gnn.model == m) & (gnn.graph == g)]
                         .assign(method=f"{m.upper()}-{g[-1].upper()} mặc định (4.1a)"))
    parts.append(tuned.assign(method="GCN-A tinh chỉnh (4.1b)"))
    parts.append(mine.assign(method=f"GGNN-B tinh chỉnh {WINNER} (4.2)"))
    allr = pd.concat(parts, ignore_index=True)
    METHODS = list(dict.fromkeys(allr.method))
    table = pd.DataFrame(index=METHODS)
    for protocol in PROTOCOLS:
        sub = allr[allr.protocol == protocol].groupby("method")
        table[f"{protocol} Macro-F1"] = sub.macro_f1.apply(lambda s: f"{s.mean():.4f} ± {s.std(ddof=0):.4f}").reindex(METHODS)
        table[f"{protocol} weighted-F1"] = sub.weighted_f1.mean().reindex(METHODS).round(4)
        table[f"{protocol} MCC"] = sub.mcc.mean().reindex(METHODS).round(4)
        table[f"{protocol} accuracy"] = sub.accuracy.mean().reindex(METHODS).round(4)
    display(table)
    table.to_csv(FIG_DIR / "test_comparison.csv")
    fig, ax = plt.subplots(figsize=(12, 4.2))
    width = 0.38
    for i, protocol in enumerate(PROTOCOLS):
        sub = allr[allr.protocol == protocol].groupby("method").macro_f1
        mean, std = sub.mean().reindex(METHODS), sub.std(ddof=0).reindex(METHODS)
        ax.bar(np.arange(len(METHODS)) + (i - 0.5) * width, mean, width, yerr=std, capsize=3, label=protocol)
    ax.set_xticks(np.arange(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=30, ha="right")
    ax.set_ylabel("Macro-F1 test (trung bình ± độ lệch chuẩn)")
    ax.set_title("Macro-F1 test: GGNN-B tinh chỉnh so với các mốc")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIG_DIR / "test_macro_f1.png", bbox_inches="tight")
    plt.show()
    # Hiệu của GGNN-B tinh chỉnh với từng mốc, tính theo độ lệch chuẩn.
    me = f"GGNN-B tinh chỉnh {WINNER} (4.2)"
    rows = []
    for protocol in PROTOCOLS:
        g = allr[allr.protocol == protocol].groupby("method").macro_f1
        mu, sd = g.mean(), g.std(ddof=0)
        for other in METHODS:
            if other == me:
                continue
            delta = mu[me] - mu[other]
            rows.append({"protocol": protocol, "so với": other, "Δ Macro-F1": round(delta, 4),
                         "độ lệch chuẩn lớn hơn": round(max(sd[me], sd[other]), 4),
                         "kết luận": "trong độ lệch chuẩn" if abs(delta) < max(sd[me], sd[other])
                         else ("cao hơn" if delta > 0 else "thấp hơn")})
    display(pd.DataFrame(rows))

In [ ]:
if RUN_FINAL_TEST:
    pc_new = pd.read_csv(FINAL_OUT_DIR / "per_class.csv")
    pc_old = pd.read_csv(A / "gnn_per_class.csv")
    protocol = "project_wise"
    new = pc_new[pc_new.protocol == protocol].groupby("cwe").agg(f1=("f1", "mean"), support=("support", "first"))
    old = pc_old[(pc_old.protocol == protocol) & (pc_old.split == "test") & (pc_old.model == "ggnn")
                 & (pc_old.graph == "graph_b")].groupby("cwe").f1.mean()
    new["f1_mặc định_4.1a"] = old.reindex(new.index)
    new = new.sort_values("support", ascending=False)
    fig, ax = plt.subplots(figsize=(13, 4.2))
    x = np.arange(len(new))
    ax.bar(x - 0.2, new["f1_mặc định_4.1a"], 0.4, label="GGNN-B mặc định (4.1a)", color="#9ecae1")
    ax.bar(x + 0.2, new.f1, 0.4, label=f"GGNN-B {WINNER} (4.2)", color="#d95f02")
    ax.set_xticks(x)
    ax.set_xticklabels(new.index, rotation=70, ha="right")
    ax.set_ylabel("F1 theo CWE (trung bình 3 seed)")
    twin = ax.twinx()
    twin.plot(x, new.support, color="black", marker="o", ms=3, lw=1, label="support test")
    twin.set_ylabel("Số mẫu test")
    twin.grid(False)
    ax.set_title(f"F1 theo CWE trên test {protocol}, sắp theo support giảm dần")
    ax.legend(loc="upper right")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "per_cwe_f1.png", bbox_inches="tight")
    plt.show()
    print("Số CWE có F1 test bằng 0, 4.2:", int((new.f1 == 0).sum()), "của", len(new),
          "| 4.1a mặc định:", int((new["f1_mặc định_4.1a"] == 0).sum()))
    display(new.round(3))

In [ ]:
if RUN_FINAL_TEST:
    matrices = json.loads((FINAL_OUT_DIR / "confusion_matrices.json").read_text(encoding="utf-8"))
    labels = list(pc_new[(pc_new.protocol == "project_wise") & (pc_new.seed == SEEDS[0])].cwe)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
    for ax, protocol in zip(axes, PROTOCOLS):
        cm = np.array(matrices[f"{protocol}|{SCOPE[1]}|{SCOPE[0]}|{SEEDS[0]}|test"], dtype=float)
        norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_xlabel("Dự đoán")
        ax.set_ylabel("Thực tế")
        ax.set_title(f"Ma trận nhầm lẫn test (seed {SEEDS[0]}, chuẩn hóa theo hàng), {protocol}")
        ax.grid(False)
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "confusion_matrix.png", bbox_inches="tight")
    plt.show()

## 8. Độ nhạy: test của cả 7 cấu hình (không dùng để chọn)

Bước này chạy sau khi lựa chọn đã cố định và kết quả test chính thức đã ghi. Script nạp trọng số đã lưu của từng cấu hình không được chọn rồi chấm trên test, không train lại. Mục đích: xem thứ hạng validation có chuyển sang test không, và việc chọn cấu hình làm Macro-F1 test dao động bao nhiêu. Kết quả chính thức của 4.2 vẫn là cấu hình đã cố định ở mục 4. Không dùng bảng này để đổi lựa chọn.

In [ ]:
if RUN_FINAL_TEST and RUN_SENSITIVITY:
    from src.training.gcn_tuning import validate_artifacts
    SENS_DIR.mkdir(parents=True, exist_ok=True)
    frames = [mine.assign(config=WINNER)]
    for k in IDS:
        if k == WINNER:
            continue
        target = SENS_DIR / k
        if target.exists() and any(target.iterdir()):
            validate_artifacts(target)  # đã chấm xong ở lần chạy trước
        else:
            run_script(["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                        "--frozen-selection", str(FROZEN), "--results-root", str(FROZEN_ROOT),
                        "--sensitivity-config", k], SENS_DIR / f"{k}.log")
        frames.append(pd.read_csv(target / "results.csv").assign(config=k))
    sens = pd.concat(frames, ignore_index=True)
    assert set(sens.split) == {"test"} and len(sens) == 6 * len(IDS)
    summary = pd.DataFrame(index=IDS)
    for protocol in PROTOCOLS:
        v = val[val.protocol == protocol].groupby("config").macro_f1
        t = sens[sens.protocol == protocol].groupby("config").macro_f1
        summary[f"{protocol} validation"] = v.mean().reindex(IDS)
        summary[f"{protocol} test"] = t.mean().reindex(IDS)
        summary[f"{protocol} test std"] = t.std(ddof=0).reindex(IDS)
    summary.index = [f"{k} *" if k == WINNER else k for k in summary.index]
    print("* = cấu hình được chọn theo validation project_wise. Bảng chỉ để phân tích độ nhạy.")
    display(summary.round(4))
    summary.to_csv(FIG_DIR / "sensitivity_test_all_configs.csv")
    notes = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, protocol in zip(axes, PROTOCOLS):
        v = summary[f"{protocol} validation"]
        t = summary[f"{protocol} test"]
        ax.errorbar(v, t, yerr=summary[f"{protocol} test std"], fmt="o", color="#7570b3", capsize=3)
        for name in summary.index:
            ax.annotate(name, (v[name], t[name]), textcoords="offset points", xytext=(5, 5),
                        color="#d95f02" if name.endswith("*") else "black")
        rho = v.corr(t, method="spearman")
        ax.set_title(f"{protocol}: validation so với test, Spearman = {rho:.2f}")
        ax.set_xlabel("Macro-F1 validation (trung bình 3 seed)")
        ax.set_ylabel("Macro-F1 test (trung bình ± độ lệch chuẩn)")
        best = t.idxmax()
        notes.append({"protocol": protocol, "Spearman validation-test": round(rho, 3),
                      "test của cấu hình được chọn": round(t[f"{WINNER} *"], 4),
                      "test cao nhất": round(t.max(), 4), "cấu hình có test cao nhất": best,
                      "khoảng dao động test giữa các cấu hình": round(t.max() - t.min(), 4)})
    plt.tight_layout()
    fig.savefig(FIG_DIR / "sensitivity_validation_vs_test.png", bbox_inches="tight")
    plt.show()
    display(pd.DataFrame(notes))

## 9. Đóng gói để nộp

Gom tệp nhẹ (không có trọng số `.pt`) vào `submission_4_2` và tệp zip cùng tên trong `/kaggle/working`. Sau khi chạy xong: bấm **Save Version** (Save & Run All) để lưu bản notebook chứa log, rồi tải tệp zip và notebook đã chạy từ tab Output.

In [ ]:
if RUN_TUNING or RUN_FINAL_TEST:
    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    (SUBMISSION_DIR / "tuning").mkdir(parents=True)
    light = shutil.ignore_patterns("*.pt")
    for item in sorted(FROZEN_ROOT.iterdir()):
        if item.is_dir():
            shutil.copytree(item, SUBMISSION_DIR / "tuning" / item.name, ignore=light)
        elif item.suffix in {".json", ".log"}:
            shutil.copy2(item, SUBMISSION_DIR / "tuning" / item.name)
    if RUN_FINAL_TEST:
        shutil.copytree(FINAL_OUT_DIR, SUBMISSION_DIR / "final", ignore=light)
        final_log = FINAL_OUT_DIR.parent / (FINAL_OUT_DIR.name + ".log")
        if final_log.exists():
            shutil.copy2(final_log, SUBMISSION_DIR / "final" / final_log.name)
    if RUN_FINAL_TEST and RUN_SENSITIVITY:
        shutil.copytree(SENS_DIR, SUBMISSION_DIR / "sensitivity", ignore=light)
    shutil.copytree(FIG_DIR, SUBMISSION_DIR / "figures")
    archive = shutil.make_archive(str(SUBMISSION_DIR), "zip", SUBMISSION_DIR)
    size = Path(archive).stat().st_size / 1e6
    print(f"Đã tạo {archive} ({size:.1f} MB)")
    for p in sorted(SUBMISSION_DIR.rglob("*")):
        if p.is_file():
            print(" ", p.relative_to(SUBMISSION_DIR))